# บทที่ 3 — ไม่มีเฉลย แล้วจะสอนโมเดลยังไง

<sub>บทเรียนที่ 3 จาก 8 &nbsp;·&nbsp; [← บทที่ 2](02_feature_extraction.ipynb) · [สารบัญ](README.md) · [บทที่ 4 →](04_data_preparation.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- เข้าใจว่าทำไมข้อมูลชุดนี้ถึงไม่มี 'คำตอบ' ให้โมเดลเรียน
- รู้จักวิธีสร้าง label 3 แบบ และข้อดีข้อเสียของแต่ละแบบ
- สร้าง Health Index ด้วยตัวเอง
- รู้ว่าการสร้าง label เองมีข้อจำกัดอะไรที่ต้องระวัง

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 3.1 ปัญหาที่หลายคนมองข้าม

การเทรนโมเดลแบบ **supervised learning** ต้องมีคู่ของ (ข้อมูลเข้า, คำตอบ)
เราได้ข้อมูลเข้าแล้วจากบทที่ 2 — แต่**คำตอบล่ะ?**

ข้อมูล IMS บอกเราแค่ว่า *"เมื่อจบการทดลอง Bearing 3 กับ 4 พัง"*
มันไม่ได้บอกว่า ณ จุดเวลาที่ 500 ลูกปืนเหลืออายุอีกกี่ชั่วโมง

**เราจึงต้องสร้างคำตอบขึ้นมาเอง** และวิธีที่เลือกส่งผลต่อทุกอย่างที่ตามมา

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
from src.paths import FEATURES_CACHE

features = np.load(FEATURES_CACHE)["features"]
N = len(features)
print("จำนวนจุดเวลา:", N)

## 3.2 ทางเลือกที่ 1 — Linear RUL

**แนวคิดที่ง่ายที่สุด:** สมมติว่าสุขภาพลดลงเป็นเส้นตรงจาก 1 (ใหม่เอี่ยม) ไป 0 (พัง)

RUL ย่อมาจาก *Remaining Useful Life* = อายุการใช้งานที่เหลือ

In [ ]:
from src.data_loader import compute_linear_rul, compute_piecewise_rul

linear = compute_linear_rul(N)

plt.figure(figsize=(11, 3.5))
plt.plot(linear, color="#d32f2f", linewidth=2)
plt.xlabel("จุดเวลา")
plt.ylabel("RUL")
plt.title("ทางเลือกที่ 1: Linear RUL - สมมติว่าเสื่อมเป็นเส้นตรง")
plt.tight_layout()
plt.show()

**ปัญหา:** เส้นนี้บอกว่าตอนกลางการทดลอง ลูกปืน "เสื่อมไปครึ่งหนึ่งแล้ว"

แต่จากบทที่ 2 เรารู้แล้วว่า RMS แทบไม่ขยับเลยจนถึงช่วงท้าย —
ลูกปืนยังปกติดีอยู่เลย!

เท่ากับเรากำลัง**สอนสิ่งที่ไม่จริง**ให้โมเดล ผลคือมันจะสับสน
เพราะข้อมูลเข้าบอกว่า "ปกติ" แต่คำตอบบอกว่า "เสื่อมครึ่งแล้ว"

## 3.3 ทางเลือกที่ 2 — Piecewise Linear

**ปรับปรุง:** ให้คงที่ที่ 1.0 ช่วงแรก แล้วค่อยลดลงเป็นเส้นตรงช่วงท้าย

ใกล้ความจริงขึ้นเยอะ แต่มีคำถามที่ตอบยาก: **จุดเปลี่ยนอยู่ตรงไหน?**

In [ ]:
plt.figure(figsize=(11, 3.5))
for ratio, c in [(0.60, "#1976d2"), (0.75, "#388e3c"), (0.90, "#f57c00")]:
    plt.plot(compute_piecewise_rul(N, usable_life_ratio=ratio),
             label=f"จุดเปลี่ยนที่ {int(ratio*100)}%", color=c, linewidth=1.8)

plt.xlabel("จุดเวลา")
plt.ylabel("RUL")
plt.title("ทางเลือกที่ 2: Piecewise Linear - แต่จุดเปลี่ยนควรอยู่ตรงไหน?")
plt.legend()
plt.tight_layout()
plt.show()

สามเส้นนี้ให้คำตอบต่างกันมาก และเรา**เดาเอาเอง**ทั้งหมด

ถ้าเดาผิด โมเดลก็เรียนผิด — และเราไม่มีทางรู้ว่าเดาถูกหรือเปล่า

## 3.4 ทางเลือกที่ 3 — Health Index (ที่เราเลือกใช้)

**แนวคิด:** แทนที่จะเดารูปร่างของเส้น ให้**อ่านจากสัญญาณจริง**

ขั้นตอน:
1. เอา feature ที่ไวต่อการเสื่อมที่สุด (RMS, Kurtosis, Crest) ของทั้ง 4 ลูกปืน
2. ปรับสเกลแต่ละตัวให้อยู่ในช่วง 0–1 แล้วบวกรวมเป็น "คะแนนความเสื่อม"
3. เกลี่ยสัญญาณรบกวนด้วยค่าเฉลี่ยเคลื่อนที่
4. กลับด้าน: เสื่อมมาก = สุขภาพน้อย

มาเขียนเองทีละขั้นเพื่อให้เห็นภาพ

In [ ]:
# ── ขั้นที่ 1-2: รวม feature ที่ไวต่อการเสื่อม ──
FEATURE_NAMES = ["RMS", "Peak", "P2P", "Crest", "Kurtosis", "Skewness",
                 "Shape", "Impulse", "Margin", "Std",
                 "BandLow", "BandMid", "BandHigh", "Centroid"]

def idx(bearing, name):
    return bearing * 14 + FEATURE_NAMES.index(name)


def normalize(v):
    return (v - v.min()) / (v.max() - v.min() + 1e-12)


degradation = np.zeros(N)
for name in ["RMS", "Kurtosis", "Crest"]:
    for bearing in range(4):
        degradation += normalize(features[:, idx(bearing, name)])

degradation /= degradation.max()      # ปรับให้สูงสุด = 1

plt.figure(figsize=(11, 3.5))
plt.plot(degradation, color="#7b1fa2", linewidth=0.9)
plt.xlabel("จุดเวลา")
plt.ylabel("คะแนนความเสื่อม")
plt.title("ขั้นที่ 1-2: คะแนนความเสื่อมดิบ (ยังมีสัญญาณรบกวน)")
plt.tight_layout()
plt.show()

เห็นได้ว่าเส้นกระโดดขึ้นลงเยอะ นั่นคือ**สัญญาณรบกวนจากการวัด** —
การวัดแต่ละครั้งมีความคลาดเคลื่อนตามธรรมชาติ

ถ้าเอาเส้นนี้ไปเป็นคำตอบเลย โมเดลจะสับสนเพราะข้อมูลเข้าที่คล้ายกัน
กลับมีคำตอบต่างกันมาก จึงต้องเกลี่ยก่อน

In [ ]:
# ── ขั้นที่ 3: เกลี่ยด้วยค่าเฉลี่ยเคลื่อนที่ ──
import pandas as pd

WINDOW = 7   # เฉลี่ย 7 จุด = 70 นาที

smoothed = pd.Series(degradation).rolling(window=WINDOW, min_periods=1, center=True).mean().values

plt.figure(figsize=(11, 3.5))
plt.plot(degradation, color="#bbb", linewidth=0.7, label="ก่อนเกลี่ย")
plt.plot(smoothed, color="#7b1fa2", linewidth=1.8, label=f"หลังเกลี่ย (window={WINDOW})")
plt.xlabel("จุดเวลา")
plt.ylabel("คะแนนความเสื่อม")
plt.title("ขั้นที่ 3: เกลี่ยสัญญาณรบกวน")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── ขั้นที่ 4: กลับด้านเป็นคะแนนสุขภาพ ──
health_index = np.clip(1.0 - smoothed, 0.0, 1.0)

plt.figure(figsize=(11, 3.5))
plt.plot(health_index, color="#2e7d32", linewidth=2)
plt.xlabel("จุดเวลา")
plt.ylabel("Health Index")
plt.title("ผลลัพธ์: Health Index (สูง = แข็งแรง, ต่ำ = ใกล้พัง)")
plt.tight_layout()
plt.show()

print(f"ช่วงค่า: [{health_index.min():.4f}, {health_index.max():.4f}]")

In [ ]:
# เทียบกับของจริงใน src/ ว่าตรงกันไหม
from src.data_loader import compute_health_index

official = compute_health_index(features, window=7)

print("\nค่าต่างกันมากที่สุด:", np.abs(official - health_index).max())
print("ตรงกัน!" if np.allclose(official, health_index, atol=1e-6) else "ไม่ตรง - ลองไล่ดูว่าต่างตรงไหน")

## 3.5 เทียบทั้ง 3 แบบ

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.plot(compute_linear_rul(N), label="1. Linear RUL", color="#d32f2f", linewidth=1.6, linestyle="--")
plt.plot(compute_piecewise_rul(N, 0.75), label="2. Piecewise (75%)", color="#f57c00", linewidth=1.6, linestyle="-.")
plt.plot(health_index, label="3. Health Index (เลือกใช้)", color="#2e7d32", linewidth=2.2)
plt.xlabel("จุดเวลา")
plt.ylabel("ค่า label")
plt.title("เทียบวิธีสร้าง label ทั้ง 3 แบบ")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# วัดว่า label แต่ละแบบสอดคล้องกับสัญญาณจริง (RMS ของลูกปืนที่พัง) แค่ไหน
from scipy.stats import pearsonr

rms_b1 = features[:, idx(0, "RMS")]

print("ความสัมพันธ์กับ RMS ของ Bearing 1 (ตัวที่พังจริง):")
print("-" * 52)
for name, lab in [("Linear RUL", compute_linear_rul(N)),
                  ("Piecewise 75%", compute_piecewise_rul(N, 0.75)),
                  ("Health Index", health_index)]:
    r, _ = pearsonr(rms_b1, lab)
    print(f"  {name:<16} r = {r:+.4f}")

ค่า r ที่**ติดลบมาก** = label ลดลงเมื่อ RMS เพิ่มขึ้น ซึ่งคือสิ่งที่เราต้องการ
(RMS สูง = เสื่อม = สุขภาพต่ำ)

Health Index สอดคล้องกับสัญญาณจริงมากที่สุด เพราะมันถูกสร้างมาจากสัญญาณนั่นเอง

## 3.6 ⚠️ ข้อจำกัดที่ต้องพูดให้ชัด

**และนี่คือจุดที่ต้องซื่อสัตย์กับตัวเอง**

Health Index ถูกคำนวณจาก RMS, Kurtosis, Crest Factor
ซึ่ง**เป็น feature ชุดเดียวกับที่เราจะป้อนเข้าโมเดล**

แปลว่าคำตอบกับข้อมูลเข้ามีความสัมพันธ์กันอยู่ก่อนแล้วโดยธรรมชาติ
โมเดลจึงทำนายได้แม่นกว่าที่ควรจะเป็น

**ผลที่ตามมา:**
- ตัวเลขความแม่นยำที่ได้ **ห้ามเอาไปเทียบตรง ๆ** กับงานที่มีข้อมูลอายุจริง
- แต่ยัง**เทียบระหว่างโมเดลด้วยกันเองได้** เพราะทุกโมเดลเจอเงื่อนไขเดียวกัน

นี่คือเหตุผลที่บทที่ 7 จะสอนวิธีอ่านผลอย่างมีวิจารณญาณ

> **บทเรียนที่ใช้ได้กับทุกโปรเจกต์ ML:** ถ้า label ถูกสร้างจากข้อมูลเข้า
> ต้องระบุข้อจำกัดนี้ทุกครั้งที่รายงานผล ไม่ใช่ซ่อนไว้

## 🔧 ลองแก้ดู — ทดลองกับ label


1. เปลี่ยน `WINDOW = 7` เป็น `WINDOW = 1` (ไม่เกลี่ยเลย) และ `WINDOW = 30` (เกลี่ยหนักมาก)
   แล้วดูรูปร่างเส้น — เกลี่ยมากไปจะเสียอะไร?
2. ลองสร้าง Health Index โดยใช้ **เฉพาะ RMS** (ตัด Kurtosis กับ Crest ออก)
   เส้นเปลี่ยนไปอย่างไร?
3. ลองใช้เฉพาะลูกปืนที่พังจริง (Bearing 3 และ 4) แทนที่จะใช้ทั้ง 4 ตัว
   — คิดว่าแบบไหนสมเหตุสมผลกว่ากัน เพราะอะไร?

## ❓ เช็คความเข้าใจ

**1. ทำไม Linear RUL ถึงทำให้โมเดลทำนายสวนทาง?**

<details>
<summary>ดูเฉลย</summary>

เพราะมันสอนว่าสุขภาพลดลงเรื่อย ๆ ตั้งแต่ต้น แต่ข้อมูลเข้าจริง ๆ แทบไม่เปลี่ยนในช่วงนั้น โมเดลจึงถูกบังคับให้หาความสัมพันธ์ที่ไม่มีอยู่จริง สุดท้ายมันเรียนรู้แค่ 'ยิ่งเวลาผ่านไป ค่ายิ่งต่ำ' ซึ่งไม่ได้อิงกับสภาพจริงของลูกปืนเลย

</details>

**2. การเกลี่ยด้วย rolling mean ทำให้เสียข้อมูลอะไรไปบ้าง?**

<details>
<summary>ดูเฉลย</summary>

เสียรายละเอียดการเปลี่ยนแปลงแบบฉับพลัน ถ้าลูกปืนพังกะทันหันภายใน 1-2 จุดเวลา การเกลี่ย 7 จุดจะทำให้เหตุการณ์นั้นถูกเฉลี่ยจนดูค่อยเป็นค่อยไป จึงต้องเลือกขนาด window ให้สมดุลระหว่างการลดสัญญาณรบกวนกับการรักษาความคมของเหตุการณ์

</details>

**3. ถ้าเรามีข้อมูลอายุการใช้งานจริง (รู้ว่าลูกปืนพังเมื่อไร) ควรใช้ label แบบไหน?**

<details>
<summary>ดูเฉลย</summary>

ควรใช้ RUL จริงเป็น label เพราะเป็นสิ่งที่เราอยากทำนายจริง ๆ และไม่มีปัญหาความสัมพันธ์ ระหว่าง label กับ input โดยอาจใช้รูปแบบ piecewise ที่จุดเปลี่ยนมาจากการวิเคราะห์จริง ไม่ใช่การเดา

</details>

---

## สรุปบทนี้

- ข้อมูล IMS ไม่มีเฉลยมาให้ เราต้องสร้าง label ขึ้นมาเอง
- Linear RUL ขัดกับสัญญาณจริง / Piecewise ต้องเดาจุดเปลี่ยน
- Health Index อ่านจากสัญญาณจริงจึงสอดคล้องกับสภาพลูกปืนมากที่สุด
- แต่เพราะสร้างจาก feature ชุดเดียวกับ input จึงต้องระบุข้อจำกัดนี้ทุกครั้งที่รายงานผล

[← บทที่ 2](02_feature_extraction.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 4 — เตรียมข้อมูล →](04_data_preparation.ipynb)**